# 09 — Benchmark C03 pareado
Closed-set por fragmentos. As 35 inferências são pré-requisitos; este notebook apenas valida e compara.
ICs individuais: Wilson. Deltas: bootstrap pareado por espécie (2.000 repetições, seed 42). Uma realização de ruído por taxa; sem estimativa de variabilidade entre treinamentos.
all/overlap/no_exact_overlap não demonstram independência genômica. Smoke é exclusivamente sintético.

In [ ]:
import os
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import json
import numpy as np
import pandas as pd
from viral_cnn.m02_workflow import begin_m02
from viral_cnn.baselines.overlap import inspect_c03_package, audit_exact_overlap
from viral_cnn.baselines.benchmark_plotting import benchmark_summary, paired_differences, render_curves, render_differences
from viral_cnn.baselines.paired import paired_discordance

run = begin_m02("short/09")
ctx = run.ctx
run_id = run.run_id
output_dir = run.output_dir
print(f"Run: {run_id} | Smoke: {ctx.is_smoke_test} | Output: {output_dir}")


In [ ]:
with run.guard():
    methods = ["resnet50_species", "resnet_noise_species", "convnext_zero_species", "convnext_transfer_species", "convnext_noise_species", "Kraken2-flat", "BLASTN"]
    noise_levels = [0.0, 0.02, 0.05, 0.10, 0.15]
    benchmark_artifacts = None
    method_records = []
    if ctx.is_smoke_test:
        rng = np.random.default_rng(42)
        train_seqs = ["".join(rng.choice(list("ACGT"), size=512)) for _ in range(20)]
        queries = train_seqs + ["".join(rng.choice(list("ACGT"), size=512)) for _ in range(20)]
        df_test_queries = pd.DataFrame({"query_id": [f"fixture_q{i}" for i in range(40)], "sequence": queries, "label": [i % 5 for i in range(40)]})
        audit_df, strata_masks = audit_exact_overlap(df_test_queries, train_seqs)
        y_true_all = audit_df.label.to_numpy()
        predictions = {m: {} for m in methods}
        for method in methods:
            for rate in noise_levels:
                pred = y_true_all.copy()
                wrong = rng.random(40) < .15 + rate
                pred[wrong] = (pred[wrong] + 1) % 5
                if method in ("Kraken2-flat", "BLASTN"):
                    pred[wrong & (rng.random(40) < rate)] = -1
                predictions[method][rate] = pred
        source_dir = None
    else:
        selected = run.configuration.get("c03_prepared_dir")
        if not selected:
            raise FileNotFoundError("Select C03_PREPARED_DIR or the explicit C03 dependency in RUN_PLAN")
        source_dir = Path(selected).expanduser().resolve()
        inspection = inspect_c03_package(source_dir, include_artifacts=True)
        if inspection["status"] != "READY":
            raise ValueError(f"C03 preflight BLOCKED: {inspection.get('error')}")
        benchmark_artifacts = inspection.pop("artifacts")
        (output_dir / "c03_preflight.json").write_text(json.dumps(inspection, indent=2))
        audit_df = benchmark_artifacts.audit_df
        strata_masks = benchmark_artifacts.strata_masks
        predictions = benchmark_artifacts.predictions
        method_records = benchmark_artifacts.method_manifests
        y_true_all = audit_df.label.to_numpy()
        print(f"C03: {source_dir} | SHA-256: {inspection['preparation_manifest']['sha256']} | conditions: {len(method_records)}")

    total_q = len(audit_df)
    overlap_q = int(strata_masks["overlap"].sum())
    no_overlap_q = int(strata_masks["no_exact_overlap"].sum())
    audit_df.to_csv(output_dir / "exact_overlap_audit.csv", index=False)


In [ ]:
with run.guard():
    df_stratified = benchmark_summary(y_true_all, predictions, strata_masks)
    # Top-5 is taken from the same validated predictions when available.
    if benchmark_artifacts is not None:
        from viral_cnn.baselines.paired import evaluate_predictions
        for idx, row in df_stratified.iterrows():
            mask = strata_masks[row["stratum"]]
            candidates = benchmark_artifacts.top_candidates[row["method"]][row["noise_rate"]]
            if candidates is not None:
                metric = evaluate_predictions(y_true_all[mask], predictions[row["method"]][row["noise_rate"]][mask], top_candidates=candidates[mask])
                df_stratified.loc[idx, "top5"] = metric["top5"]
    df_stratified.to_csv(output_dir / "benchmark_metrics_stratified.csv", index=False)
    df_bootstrap = paired_differences(y_true_all, predictions, strata_masks)
    df_bootstrap.to_csv(output_dir / "species_cluster_bootstrap.csv", index=False)
    rows = []
    for a, b in df_bootstrap[["method_a", "method_b"]].drop_duplicates().itertuples(index=False, name=None):
        for rate in noise_levels:
            for stratum, mask in strata_masks.items():
                rows.append({"method_a": a, "method_b": b, "noise_rate": rate, "stratum": stratum,
                             **paired_discordance(y_true_all[mask], predictions[a][rate][mask], predictions[b][rate][mask])})
    pd.DataFrame(rows).to_csv(output_dir / "paired_discordance.csv", index=False)
    print(df_stratified[["method", "noise_rate", "stratum", "n", "accuracy", "coverage", "conditional_accuracy"]].head(7).to_string(index=False))


In [ ]:
with run.guard():
    figure_records = render_curves(output_dir / "benchmark_metrics_stratified.csv", output_dir)
    figure_records.append(render_differences(output_dir / "species_cluster_bootstrap.csv", output_dir))
    (output_dir / "figure_sources.json").write_text(json.dumps(figure_records, indent=2))
    inputs = [source_dir] if source_dir else []
    checkpoints = {record["model_variant"]: record["checkpoint"]["path"] for record in method_records if record.get("model_variant")}
    # References/checkpoints can live outside the prepared directory.
    for record in method_records:
        for ref in [record.get("reference"), record.get("query_manifest")]:
            if isinstance(ref, dict) and ref.get("path"):
                inputs.append(Path(ref["path"]))
                if isinstance(ref.get("database_build_manifest"), dict):
                    inputs.append(Path(ref["database_build_manifest"]["path"]))
    manifest = run.complete(input_files=inputs, checkpoints=checkpoints)
    print(f"Manifest: {run.manifest_path}")
